In [27]:
from pyspark.sql import SparkSession
import json
import os
from pyspark.sql import functions as sf
from pyspark.sql.window import Window
from delta.tables import DeltaTable

ACCESS_KEY = os.environ.get("AWS_ACCESS_KEY_ID", "forge-commerce-user")
SECRET_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "forge-commerce-pass")
S3_ENDPOINT = os.environ.get("AWS_S3_ENDPOINT", "http://minio:9000")
ORDERS = "orders"
ORDER_ITEMS = "order_items"
PRODUCTS = "products"
CUSTOMERS = "customers"
PAYMENTS = "payments"
RAW_BUCKET = "raw"
CLEANED_BUCKET = "cleaned"
CURATED_BUCKET = "curated"
RAW_PATH = f"s3a://{RAW_BUCKET}/"
CLEANED_PATH = f"s3a://{CLEANED_BUCKET}/"
CURATED_PATH = f"s3a://{CURATED_BUCKET}/"

In [28]:
spark = (
        SparkSession.builder.appName("test_payments")
        .master(os.environ.get("SPARK_MASTER", "spark://spark-master:7077"))
        .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY)
        .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY)
        .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        # Delta Lake configurations
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
        .getOrCreate()
    )

In [29]:
payments = spark.read.format("delta").load(CURATED_PATH + PAYMENTS)
payments.printSchema()

root
 |-- payment_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_amount: decimal(10,2) (nullable = true)
 |-- net_amount: decimal(10,2) (nullable = true)
 |-- transaction_fee: decimal(12,2) (nullable = true)
 |-- transaction_fee_rate: decimal(10,2) (nullable = true)
 |-- chargeback_amount: decimal(12,2) (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_gateway: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- payment_reference: string (nullable = true)
 |-- payment_uuid: string (nullable = true)
 |-- payment_date: string (nullable = true)
 |-- payment_time: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- payment_timestamp: timestamp (nullable = true)
 |-- chargeback_date: date (nullable = true)
 |-- chargeback_reason: string (nullable = tru

In [30]:
# get customer id with more payments
customer_id = payments.groupBy("customer_id").count().orderBy("count", ascending=False).limit(1).first().customer_id
print(customer_id)

# order payemnts for that customer id
payments.filter(payments.customer_id == customer_id) \
    .orderBy(sf.col("created_at").asc()) \
    .select("payment_id", "customer_ID", "sk_customer", "created_at") \
    .show()

641


+----------+-----------+-----------+-------------------+
|payment_id|customer_ID|sk_customer|         created_at|
+----------+-----------+-----------+-------------------+
|       150|        641|       3923|2026-03-28 21:30:27|
|       334|        641|       3923|2026-03-28 21:30:28|
|       581|        641|       3923|2026-03-28 21:30:28|
|       526|        641|       3923|2026-03-28 21:30:28|
|       646|        641|       3923|2026-03-28 21:30:28|
|       713|        641|       3923|2026-03-28 21:30:28|
|       757|        641|       3923|2026-03-28 21:30:29|
|      1150|        641|       3923|2026-03-30 21:37:19|
|      1334|        641|       3923|2026-03-30 21:37:20|
|      1581|        641|       3923|2026-03-30 21:37:20|
|      1526|        641|       3923|2026-03-30 21:37:20|
|      1713|        641|       3923|2026-03-30 21:37:20|
|      1646|        641|       3923|2026-03-30 21:37:20|
|      1757|        641|       3923|2026-03-30 21:37:20|
|      2103|        641|       

In [31]:
spark.read.format("delta").load(CURATED_PATH + CUSTOMERS) \
    .filter(sf.col("customer_id") == customer_id) \
    .orderBy(sf.col("created_at").asc()) \
    .select("customer_id", "sk_customer", "effective_from", "effective_to", "is_active") \
    .show()

+-----------+-----------+-------------------+-------------------+---------+
|customer_id|sk_customer|     effective_from|       effective_to|is_active|
+-----------+-----------+-------------------+-------------------+---------+
|        641|       1281|2026-03-28 21:22:12|2026-03-28 21:22:20|    false|
|        641|       1282|2026-03-28 21:22:20|2026-03-28 21:30:13|    false|
|        641|       3922|2026-03-28 21:30:13|2026-03-28 21:30:23|    false|
|        641|       3923|2026-03-28 21:30:23|2026-03-30 21:37:26|    false|
|        641|       6922|2026-03-30 21:37:26|2026-03-30 21:38:01|    false|
|        641|       6923|2026-03-30 21:38:01|               NULL|     true|
+-----------+-----------+-------------------+-------------------+---------+



In [32]:
# count distinct payment_ids
print(payments.select("payment_id").distinct().count())

# count distinct payment rows
print(payments.distinct().count())



5000


5000


In [33]:
spark.stop()